# Q10: KL Divergence
**Topic:** Coding-question | **Difficulty:** Medium | **Time:** 15 min
**Sheet:** Grind75ML

---

## Question
Implement KL Divergence from scratch and demonstrate it on discrete and continuous distributions.

## Theory

**KL Divergence** (Kullback-Leibler Divergence) measures information lost when distribution $Q$ is used to approximate $P$:

**Discrete:** $D_{KL}(P \| Q) = \sum_{x} P(x) \log \frac{P(x)}{Q(x)}$

**Continuous:** $D_{KL}(P \| Q) = \int P(x) \log \frac{P(x)}{Q(x)} dx$

**Properties:**
- Always $\geq 0$ (Gibbs' inequality)
- $= 0$ if and only if $P = Q$ almost everywhere
- **Not symmetric** and **not a metric** (doesn't satisfy triangle inequality)
- **Forward KL** ($D_{KL}(P\|Q)$): mean-seeking, penalizes Q=0 where P>0
- **Reverse KL** ($D_{KL}(Q\|P)$): mode-seeking, used in variational inference

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def kl_divergence_discrete(p: np.ndarray, q: np.ndarray, eps: float = 1e-12) -> float:
    """KL Divergence for discrete distributions.
    
    D_KL(P || Q) = sum(P * log(P / Q))
    """
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    # Clip for numerical stability
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    return np.sum(p * np.log(p / q))


def kl_divergence_gaussian(mu1: float, sigma1: float, 
                           mu2: float, sigma2: float) -> float:
    """Analytical KL Divergence between two univariate Gaussians.
    
    D_KL(N1 || N2) = log(σ2/σ1) + (σ1² + (μ1-μ2)²) / (2σ2²) - 1/2
    """
    return (np.log(sigma2 / sigma1) + 
            (sigma1**2 + (mu1 - mu2)**2) / (2 * sigma2**2) - 0.5)


# --- Discrete Example ---
p = np.array([0.4, 0.3, 0.2, 0.1])
q1 = np.array([0.25, 0.25, 0.25, 0.25])  # uniform
q2 = np.array([0.35, 0.30, 0.20, 0.15])  # close to p

print("=== Discrete KL Divergence ===")
print(f"D_KL(P || Uniform) = {kl_divergence_discrete(p, q1):.6f}")
print(f"D_KL(P || Close)   = {kl_divergence_discrete(p, q2):.6f}")
print(f"D_KL(P || P)       = {kl_divergence_discrete(p, p):.6f}")  # should be ~0

# Asymmetry demonstration
print(f"\nD_KL(P || Q1) = {kl_divergence_discrete(p, q1):.6f}")
print(f"D_KL(Q1 || P) = {kl_divergence_discrete(q1, p):.6f}")
print("Note: These are different! KL is asymmetric.")

In [ ]:
# --- Gaussian Example ---
print("\n=== Gaussian KL Divergence ===")
kl_gauss = kl_divergence_gaussian(mu1=0, sigma1=1, mu2=1, sigma2=2)
print(f"D_KL(N(0,1) || N(1,4)) = {kl_gauss:.6f}")

# Visualize
x = np.linspace(-6, 8, 1000)
p_dist = (1/(np.sqrt(2*np.pi)*1)) * np.exp(-0.5*((x-0)/1)**2)
q_dist = (1/(np.sqrt(2*np.pi)*2)) * np.exp(-0.5*((x-1)/2)**2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, p_dist, 'b-', linewidth=2, label='P ~ N(0, 1)')
ax.plot(x, q_dist, 'r--', linewidth=2, label='Q ~ N(1, 4)')
ax.fill_between(x, p_dist, alpha=0.2, color='blue')
ax.fill_between(x, q_dist, alpha=0.2, color='red')
ax.set_title(f'KL Divergence = {kl_gauss:.4f}', fontsize=14)
ax.legend(fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('Density')
plt.tight_layout()
plt.show()

## Key Takeaways
- KL Divergence is the **extra bits** needed to encode data from P using a code optimized for Q
- **Forward KL** (used in MLE) covers all modes → avoids missing probability mass
- **Reverse KL** (used in variational inference) focuses on a single mode → avoids placing mass where P is low
- For Gaussians, KL has a **closed-form** — no integration needed